In [326]:
import numpy as np
import matplotlib.pyplot as plt
from SNNLayer import SNNLayer

In [327]:
dt = 1e-3
tau_m = 10e-3
tau_syn = 2e-3
V_th = 0.5
V_reset = 0.0
R = 10
gamma = 0.95 #np.exp(-dt / tau_syn)
alpha = 1 - dt / tau_m
beta = R * dt / tau_m

In [328]:
W_input_layer = np.load("W_input_layer.npy")
W_output_layer = np.load("W_output_layer.npy")

In [329]:
input_layer = SNNLayer(4, 16, dt, gamma, tau_m, V_th)
input_layer.W = W_input_layer.T / np.max(np.abs(W_input_layer))
output_layer = SNNLayer(16, 3, dt, gamma, tau_m, V_th)
output_layer.W = W_output_layer.T / np.max(np.abs(W_output_layer))

In [330]:
print(input_layer.W, output_layer.W)

[[-0.39989188  0.36997005  0.47217628  0.18789533]
 [ 0.6301398   0.26547658  0.27860814 -0.02194869]
 [-0.8392028  -0.49278045 -0.70771575  0.45959917]
 [-0.04021491 -0.8116892  -0.8228487   0.20960711]
 [ 0.6804896  -0.02757487 -0.6153185   1.        ]
 [-0.37884423  0.8555878  -0.35277912  0.84216285]
 [-0.33959016 -0.7096661  -0.17913982 -0.10883521]
 [ 0.2828957   0.96073246  0.5388649   0.14885621]
 [ 0.45676103  0.07056825 -0.6640094  -0.06166459]
 [-0.12815218  0.7900146   0.0685563  -0.17401971]
 [ 0.02327471  0.7601428   0.54066503 -0.81354   ]
 [ 0.59810483  0.45664495 -0.35861441  0.12670344]
 [-0.7892724  -0.8555126  -0.04187472 -0.8433583 ]
 [-0.9110702   0.1184948  -0.8483321  -0.7511326 ]
 [-0.58360595  0.05042583 -0.49439082  0.17829353]
 [-0.6024703  -0.08929551 -0.18791844 -0.9339101 ]] [[ 0.7958252   0.8944457   0.0848531   0.78100884 -0.69636416  0.23156327
  -0.91035885 -0.48588115  0.07057825 -0.39201173 -0.45112434 -0.12226821
   0.4596913   0.49963817  0.206255

In [331]:
def encode_spike(sample, dt, T=10, freq=50):

    spike_train = np.zeros((T, sample.shape[0]))
    for i in range(sample.shape[0]):

        p = sample * freq * dt
        spike_train[i] = np.random.rand(sample.shape[0]) < p

    return spike_train

In [332]:
sample = np.array([5.5, 0.6, 2.2, 10])
sample = sample / np.linalg.norm(sample)
spike_train = encode_spike(sample, dt)
print(spike_train.shape)
print(spike_train)

(10, 4)
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


In [333]:
T = spike_train.shape[0]

spike_hidden = []
spike_output = []

for t in range(T):

    s_h = input_layer.step(spike_train[t])
    spike_hidden.append(s_h)

spike_hidden = np.array(spike_hidden)

for t in range(T):

    s_o = output_layer.step(spike_hidden[t])
    spike_output.append(s_o)

spike_output = np.array(spike_output)

In [334]:
print(spike_hidden)
print(spike_output)
print(spike_hidden.sum(axis=0))
print(spike_output.sum(axis=0))

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0.]


In [335]:
spike_count = spike_output.sum(axis=0)
predicted_class = np.argmax(spike_count)

print(spike_output)
print(predicted_class)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
0
